# Simple Pendulum Identification with RHONN

## Overview
Neural identification of a simple pendulum using:
- **PF-RHONN**: Particle Filter (neuron-specific parameters)

## Features
- RK4 discretization
- Non-Gaussian noise models
- NaN prevention strategies
- Comprehensive error metrics
- 2-state system: angle (θ) and angular velocity (ω)

## Execution
Run cells in order: Definitions → Simulation → Analysis

In [9]:
# Neural Identifier Training - Simple Pendulum
# Method: Particle Filter (PF-RHONN)
# Con características específicas por neurona

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1) True nonlinear system (Simple Pendulum)
# ============================================================
def plant_dynamics(state, u):
    """
    Dynamics of a simple pendulum with damping and external torque.
    state = [θ, ω] 
        θ: angle from vertical downward position (rad)
        ω: angular velocity (rad/s)
    u = [τ]: external torque on joint (N⋅m)
    """
    # Physical Parameters
    m = 1.0      # Mass (kg)
    l = 1.0      # Length (m)
    g = 9.81     # Gravity (m/s²)
    b = 0.1      # Damping coefficient (N⋅m⋅s)
    
    θ, ω = state
    τ = u[0] if len(u) > 0 else 0.0
    
    # Moment of inertia: I = m·l²
    I = m * l**2
    
    # Angular acceleration: α = -(g/l)sin(θ) - (b/I)ω + τ/I
    α = -(g/l) * np.sin(θ) - (b/I) * ω + τ/I
    
    # State derivatives
    θ_dot = ω
    ω_dot = α
    
    return np.array([θ_dot, ω_dot])

def plant(x_k, u_k, dt=0.01):
    """
    RK4 integration step for better accuracy.
    """
    k1 = plant_dynamics(x_k, u_k)
    k2 = plant_dynamics(x_k + 0.5*dt*k1, u_k)
    k3 = plant_dynamics(x_k + 0.5*dt*k2, u_k)
    k4 = plant_dynamics(x_k + dt*k3, u_k)
    
    x_kp1 = x_k + (dt/6.0) * (k1 + 2*k2 + 2*k3 + k4)
    
    return x_kp1

# ============================================================
# 2) RHONN structure - CARACTERÍSTICAS POR NEURONA
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -50, 50) 
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input, neuron_index):
    """
    Features for Simple Pendulum - ESPECÍFICAS PARA CADA NEURONA.
    
    x_est = [θ, ω]
    u_input = [τ]
    neuron_index: índice de la neurona (0=θ, 1=ω)
    """
    θ, ω = x_est
    τ = u_input[0] if len(u_input) > 0 else 0.0
    
    # Términos básicos sigmoidales
    s_θ = sigmoidal(θ)
    s_ω = sigmoidal(ω)
    s_τ = sigmoidal(τ)
    
    # ========== CARACTERÍSTICAS ESPECÍFICAS POR NEURONA ==========
    
    if neuron_index == 0:  # Neurona para θ (ángulo)
        # dθ/dt = ω
        return np.array([
            # s_ω,                       # Velocidad angular (término principal)
            s_θ**3,                    # Término cuadrático
            # s_θ,                       # Posición angular
            s_θ * s_ω,                # Acoplamiento posición-velocidad
            1.0                        # Bias
        ])
    
    elif neuron_index == 1:  # Neurona para ω (velocidad angular)
        # dω/dt = -(g/l)sin(θ) - (b/I)ω + τ/I
        return np.array([
            # s_θ,                       # Término gravitacional
            # s_ω,                       # Término de fricción
            s_ω**3,                    # Fricción no lineal
            s_τ,                       # Torque externo
            s_θ * s_ω,                # Acoplamiento
            1.0                        # Bias
        ])
    
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# Función auxiliar para obtener el tamaño de características de cada neurona
def get_z_size(neuron_index):
    """Retorna el número de características para una neurona dada."""
    if neuron_index == 0:  # θ
        return 3  # [s_ω, s_ω², s_θ, s_θ*s_ω, bias]
    elif neuron_index == 1:  # ω
        return 4  # [s_θ, s_ω, s_ω², s_τ, s_θ*s_ω, bias]
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# ============================================================
# 3) Particle Filter Trainer
# ============================================================

class Generic_RHONN_Trainer:
    """ Base class to handle the loop logic easily """
    def get_prediction(self, weights, x_k, u_k, neuron_idx):
        z = construct_z_vector(x_k, u_k, neuron_idx)
        return np.dot(weights, z)


class PF_Trainer(Generic_RHONN_Trainer):
    """
    Robust Particle Filter Trainer with NaN prevention strategies.
    Now supports neuron-specific Q and R values.
    """
    def __init__(self, n_neurons, n_particles=1000, Q_std=None, R_std=None):
        self.n_neurons = n_neurons
        self.n_particles = n_particles
        
        # Initialize particles with controlled variance
        self.particles = [np.random.randn(n_particles, get_z_size(i)) * 0.1 
                         for i in range(n_neurons)]
        
        # Initialize weights uniformly
        self.weights_pf = [np.ones(n_particles) / n_particles for _ in range(n_neurons)]
        
        # Tuning parameters - neuron-specific or default
        if Q_std is None:
            self.Q_std = [0.05] * n_neurons
        elif isinstance(Q_std, (list, np.ndarray)):
            assert len(Q_std) == n_neurons, f"Q_std must have length {n_neurons}"
            self.Q_std = list(Q_std)
        else:
            self.Q_std = [Q_std] * n_neurons
        
        if R_std is None:
            self.R_std = [0.05] * n_neurons
        elif isinstance(R_std, (list, np.ndarray)):
            assert len(R_std) == n_neurons, f"R_std must have length {n_neurons}"
            self.R_std = list(R_std)
        else:
            self.R_std = [R_std] * n_neurons
        
        self.regularization_std = 0.001  # Jitter after resampling
        
        # NaN prevention parameters
        self.min_weight = 1e-300
        self.max_log_likelihood = 1000.0
        self.resample_threshold = 0.3
        
    def _normalize_weights(self, weights):
        """Safely normalize weights with NaN and underflow protection."""
        if np.any(~np.isfinite(weights)):
            print("⚠️  Warning: Non-finite weights detected, resetting to uniform")
            return np.ones_like(weights) / len(weights)
        
        weights = np.maximum(weights, self.min_weight)
        weight_sum = np.sum(weights)
        
        if weight_sum < self.min_weight or not np.isfinite(weight_sum):
            print("⚠️  Warning: Invalid weight sum, resetting to uniform")
            return np.ones_like(weights) / len(weights)
        
        return weights / weight_sum
    
    def _compute_log_likelihood(self, errors, neuron_idx):
        """Compute log-likelihood with numerical stability (Student-t distribution)."""
        errors_clipped = np.clip(errors, -1000, 1000)
        
        df = 3.0  # degrees of freedom (heavy tails)
        sigma = self.R_std[neuron_idx]
        
        normalized_errors_sq = (errors_clipped / sigma) ** 2
        log_likelihood = -((df + 1) / 2.0) * np.log(1.0 + normalized_errors_sq / df)
        
        return log_likelihood
    
    def _resample_particles(self, neuron_idx):
        """Systematic resampling with regularization."""
        weights = self.weights_pf[neuron_idx]
        particles = self.particles[neuron_idx]
        n_weights = get_z_size(neuron_idx)
        
        weights = self._normalize_weights(weights)
        
        positions = (np.arange(self.n_particles) + np.random.random()) / self.n_particles
        cumulative_sum = np.cumsum(weights)
        
        indices = np.searchsorted(cumulative_sum, positions)
        indices = np.clip(indices, 0, self.n_particles - 1)
        
        self.particles[neuron_idx] = particles[indices].copy()
        
        jitter = np.random.randn(self.n_particles, n_weights) * self.regularization_std
        self.particles[neuron_idx] += jitter
        
        self.weights_pf[neuron_idx] = np.ones(self.n_particles) / self.n_particles
    
    def update(self, x_kp1, x_k, u_k):
        """Update step with NaN prevention and neuron-specific Q/R."""
        for i in range(self.n_neurons):
            try:
                z = construct_z_vector(x_k, u_k, i)
                n_weights = get_z_size(i)
                
                if not np.all(np.isfinite(z)):
                    print(f"⚠️  Warning: Non-finite feature vector for neuron {i}, skipping update")
                    continue
                
                # Prediction: Add process noise
                drift = np.random.randn(self.n_particles, n_weights) * self.Q_std[i]
                self.particles[i] += drift
                self.particles[i] = np.clip(self.particles[i], -1000, 1000)
                
                # Update: Compute likelihoods
                preds = self.particles[i] @ z
                
                if not np.all(np.isfinite(preds)):
                    print(f"⚠️  Warning: Non-finite predictions for neuron {i}, resetting particles")
                    self.particles[i] = np.random.randn(self.n_particles, n_weights) * 0.1
                    continue
                
                errors = x_kp1[i] - preds
                log_likelihood = self._compute_log_likelihood(errors, i)
                log_likelihood -= np.max(log_likelihood)
                
                self.weights_pf[i] *= np.exp(log_likelihood)
                self.weights_pf[i] = self._normalize_weights(self.weights_pf[i])
                
                # Resampling
                weight_sq_sum = np.sum(self.weights_pf[i] ** 2)
                eff_N = 1.0 / weight_sq_sum if weight_sq_sum > 0 else 0
                
                if eff_N < self.resample_threshold * self.n_particles:
                    self._resample_particles(i)
                    
            except Exception as e:
                print(f"⚠️  Error in PF update for neuron {i}: {e}")
                self.particles[i] = np.random.randn(self.n_particles, get_z_size(i)) * 0.1
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
    
    def get_estimates(self):
        """Get weighted average of particles (with NaN protection)."""
        estimates = []
        for i in range(self.n_neurons):
            weights = self._normalize_weights(self.weights_pf[i])
            estimate = np.average(self.particles[i], axis=0, weights=weights)
            
            if not np.all(np.isfinite(estimate)):
                print(f"⚠️  Warning: Non-finite estimate for neuron {i}, using median")
                estimate = np.median(self.particles[i], axis=0)
            
            estimates.append(estimate)
        
        return estimates

# ============================================================
# 4) Error Metrics Functions
# ============================================================
def calculate_error_metrics(y_true, y_pred, metric_name="State"):
    """Calculate comprehensive error metrics."""
    errors = y_true - y_pred
    
    mae = np.mean(np.abs(errors), axis=0)
    mae_total = np.mean(mae)
    
    rmse = np.sqrt(np.mean(errors**2, axis=0))
    rmse_total = np.sqrt(np.mean(rmse**2))
    
    ranges = np.max(y_true, axis=0) - np.min(y_true, axis=0)
    ranges[ranges < 1e-10] = 1.0
    nrmse = rmse / ranges
    nrmse_total = np.mean(nrmse)
    
    max_error = np.max(np.abs(errors), axis=0)
    mse = np.mean(errors**2, axis=0)
    mse_total = np.mean(mse)
    
    ss_res = np.sum(errors**2, axis=0)
    ss_tot = np.sum((y_true - np.mean(y_true, axis=0))**2, axis=0)
    r2 = 1 - (ss_res / (ss_tot + 1e-10))
    r2_total = np.mean(r2)
    
    return {
        'MAE': mae,
        'MAE_total': mae_total,
        'RMSE': rmse,
        'RMSE_total': rmse_total,
        'NRMSE': nrmse,
        'NRMSE_total': nrmse_total,
        'MSE': mse,
        'MSE_total': mse_total,
        'Max_Error': max_error,
        'R2': r2,
        'R2_total': r2_total
    }

print("✓ Definitions loaded successfully!")

✓ Definitions loaded successfully!


In [10]:
import time

# ============================================================
# 5) Simulation Main Loop - SIMPLE PENDULUM
# ============================================================
if __name__ == "__main__":
    # Simulation parameters
    n_steps = 1000
    dt = 0.01
    t = np.linspace(0, (n_steps-1)*dt, n_steps)
    
    n_states = 2  # [θ, ω]
    
    # Measurement noise parameters
    angle_noise_std = 0.01        # 0.01 rad (~0.57°)
    omega_noise_std = 0.02        # 0.02 rad/s
    
    # Non-Gaussian noise parameters
    t_df = 3.0                    # Student's t degrees of freedom
    mixture_outlier_prob = 0.09   # 9% outlier probability
    outlier_scale = 5.0           # Outlier magnitude
    
    # ============================================================
    # Initialize weights
    # ============================================================
    np.random.seed(7517)
    initial_weights = []
    for i in range(n_states):
        w_init = np.random.uniform(-1.0, 1.0, size=get_z_size(i))
        initial_weights.append(w_init.copy())
    
    print("Initial RHONN weights (uniform distribution):")
    for i, w in enumerate(initial_weights):
        print(f"  Neuron {i}: shape={w.shape}, min={w.min():.4f}, max={w.max():.4f}")
    
    # ============================================================
    # PF-RHONN Parameters
    # ============================================================
    pf_Q_std = [0.5, 0.5]      # Process noise [θ, ω]
    pf_R_std = [1.0e-3, 1.0e-2]  # Measurement noise [θ, ω]
    
    print("\n" + "="*70)
    print("PF-RHONN Neuron-Specific Parameters:")
    print("="*70)
    state_names = ['θ (angle)', 'ω (ang vel)']
    for i in range(n_states):
        print(f"  Neuron {i} ({state_names[i]}): Q_std={pf_Q_std[i]:.4f}, R_std={pf_R_std[i]:.4f}")
    
    # Initialize Trainer
    pf = PF_Trainer(n_states, n_particles=300, Q_std=pf_Q_std, R_std=pf_R_std)
    
    for i in range(n_states):
        pf.particles[i] = np.tile(initial_weights[i], (pf.n_particles, 1))
    
    print("\n✓ Filter initialized with same RHONN weights")
    
    # State arrays
    x_true = np.zeros((n_steps, 2))
    x_est_pf = np.zeros((n_steps, 2))
    y_measured = np.zeros((n_steps, 2))
    
    pf_training_times = []
    
    # Initial conditions
    x_true[0] = [np.pi/4, 0.0]  # 45° angle, zero velocity
    x_est_pf[0] = x_true[0]
    
    initial_noise = np.array([
        np.random.standard_t(t_df) * angle_noise_std * np.sqrt((t_df-2)/t_df),
        np.random.standard_t(t_df) * omega_noise_std * np.sqrt((t_df-2)/t_df)
    ])
    y_measured[0] = x_true[0] + initial_noise
    
    # Control input (external torque)
    u_hist = np.zeros((n_steps, 1))
    for k in range(n_steps):
        τ = 0.5 * np.sin(1.0 * t[k]) + 0.3 * np.sin(2.5 * t[k]) + 0.2 * np.cos(3.0 * t[k])
        
        if 400 < k < 450:
            τ += 1.0
        elif 900 < k < 950:
            τ -= 0.8
        elif 1200 < k < 1250:
            τ += 0.6
            
        u_hist[k] = [τ]

    print("\n" + "="*70)
    print("Simulating Simple Pendulum (PF-RHONN)...")
    print("="*70)
    print("\nFeature structure per neuron:")
    for i in range(n_states):
        print(f"  Neuron {i} ({state_names[i]}): {get_z_size(i)} features")
    
    print(f"\nConfiguration: SERIES (PF uses measurements as input)")
    print(f"Integration Method: Runge-Kutta 4th Order (RK4)")
    print(f"Noise - Angle (θ): ±{angle_noise_std:.4f} rad (~{angle_noise_std*57.3:.2f}°)")
    print(f"Noise - Angular velocity (ω): ±{omega_noise_std:.4f} rad/s")
    print(f"Outlier probability: {mixture_outlier_prob*100:.1f}% (scale: {outlier_scale}x)")
    print(f"Time step (dt): {dt} s")
    print(f"Simulation duration: {n_steps*dt:.1f} s ({n_steps} steps)")
    
    # Main simulation loop
    for k in range(n_steps - 1):
        # 1. Generate true next state
        x_true[k+1] = plant(x_true[k], u_hist[k], dt)
        
        # 2. Create noisy measurement with NON-GAUSSIAN noise
        measurement_noise = np.zeros(2)
        
        for i in range(2):
            is_outlier = np.random.random() < mixture_outlier_prob
            
            if i == 0:  # Angle (θ)
                if is_outlier:
                    measurement_noise[i] = np.random.normal(0, angle_noise_std * outlier_scale)
                else:
                    t_sample = np.random.standard_t(t_df)
                    measurement_noise[i] = t_sample * angle_noise_std * np.sqrt((t_df-2)/t_df)
            else:  # Angular velocity (ω)
                if is_outlier:
                    measurement_noise[i] = np.random.normal(0, omega_noise_std * outlier_scale)
                else:
                    t_sample = np.random.standard_t(t_df)
                    measurement_noise[i] = t_sample * omega_noise_std * np.sqrt((t_df-2)/t_df)
        
        y_measured[k+1] = x_true[k+1]  # Use clean measurements for this experiment
        
        # 3. Update PF
        start_time = time.perf_counter()
        pf.update(x_true[k+1], x_true[k], u_hist[k])
        pf_time = time.perf_counter() - start_time
        pf_training_times.append(pf_time)
        
        w_pf = pf.get_estimates()
        for i in range(2):
            z_pf = construct_z_vector(x_true[k], u_hist[k], i)
            x_est_pf[k+1, i] = np.dot(w_pf[i], z_pf)
        
        # Progress reporting
        if k % 300 == 0 and k > 0:
            print(f"Step {k}/{n_steps-1}")
            err_pf = np.linalg.norm(x_true[k] - x_est_pf[k])
            print(f"  Current error - PF: {err_pf:.4f}")

    # ============================================================
    # 6) Calculate Error Metrics
    # ============================================================
    
    print("\n" + "="*70)
    print("📊 COMPREHENSIVE RESULTS - SIMPLE PENDULUM")
    print("="*70)
    
    metrics_pf = calculate_error_metrics(x_true, x_est_pf, "PF-RHONN")
    
    print("\n--- PF-RHONN Performance Metrics ---")
    print(f"RMSE total: {metrics_pf['RMSE_total']:.6f}")
    print(f"R² total: {metrics_pf['R2_total']:.4f}")
    for i, name in enumerate(state_names):
        print(f"  {name}: RMSE={metrics_pf['RMSE'][i]:.6f}, MAE={metrics_pf['MAE'][i]:.6f}, "
              f"NRMSE={metrics_pf['NRMSE'][i]:.4f}, R²={metrics_pf['R2'][i]:.4f}")
    
    # Training time analysis
    print("\n" + "="*70)
    print("⏱️  TRAINING TIME ANALYSIS")
    print("="*70)
    
    pf_total_time = np.sum(pf_training_times)
    pf_mean_time = np.mean(pf_training_times)
    pf_std_time = np.std(pf_training_times)
    
    print(f"\nPF-RHONN:")
    print(f"  Total: {pf_total_time:.6f} s | Mean: {pf_mean_time*1000:.4f} ms")
    print(f"  Std: {pf_std_time*1000:.4f} ms | Min: {np.min(pf_training_times)*1000:.4f} ms")
    print(f"  Max: {np.max(pf_training_times)*1000:.4f} ms")
    
    # Final weights
    print("\n" + "="*70)
    print("🔧 FINAL RHONN WEIGHTS")
    print("="*70)
    
    w_pf_final = pf.get_estimates()
    for i in range(n_states):
        print(f"\nNeuron {i} ({state_names[i]}) - {get_z_size(i)} weights:")
        print(f"  {w_pf_final[i]}")
        print(f"  Norm: {np.linalg.norm(w_pf_final[i]):.4f}, Mean: {np.mean(w_pf_final[i]):.4f}")
    
    print("\n" + "="*70)
    print("✅ SIMULATION COMPLETED - SIMPLE PENDULUM")
    print("="*70)
    print(f"\nSystem: Simple Pendulum with Damping")
    print(f"Integration: Runge-Kutta 4th Order (RK4)")
    print(f"Time step: {dt} s, Duration: {n_steps*dt:.1f} s")
    print(f"States: {n_states} [θ, ω]")

Initial RHONN weights (uniform distribution):
  Neuron 0: shape=(3,), min=-0.7101, max=0.3165
  Neuron 1: shape=(4,), min=-0.6689, max=0.9859

PF-RHONN Neuron-Specific Parameters:
  Neuron 0 (θ (angle)): Q_std=0.5000, R_std=0.0010
  Neuron 1 (ω (ang vel)): Q_std=0.5000, R_std=0.0100

✓ Filter initialized with same RHONN weights

Simulating Simple Pendulum (PF-RHONN)...

Feature structure per neuron:
  Neuron 0 (θ (angle)): 3 features
  Neuron 1 (ω (ang vel)): 4 features

Configuration: SERIES (PF uses measurements as input)
Integration Method: Runge-Kutta 4th Order (RK4)
Noise - Angle (θ): ±0.0100 rad (~0.57°)
Noise - Angular velocity (ω): ±0.0200 rad/s
Outlier probability: 9.0% (scale: 5.0x)
Time step (dt): 0.01 s
Simulation duration: 10.0 s (1000 steps)
Step 300/999
  Current error - PF: 0.0026
Step 600/999
  Current error - PF: 0.0008
Step 900/999
  Current error - PF: 0.0036

📊 COMPREHENSIVE RESULTS - SIMPLE PENDULUM

--- PF-RHONN Performance Metrics ---
RMSE total: 0.002574
R² tot

In [11]:
# ============================================================
# 7) STATE TRAJECTORY PLOTS (SEPARATE FIGURES)
# ============================================================

# Thesis-style configuration
thesis_config = {
    'font_family': 'Computer Modern',
    'font_size': 14,
    'title_size': 16,
    'gridcolor': 'rgba(128,128,128,0.2)',
    'plot_bgcolor': 'white',
    'paper_bgcolor': 'white'
}

# θ trajectory figure
fig_theta = go.Figure()
fig_theta.add_trace(go.Scatter(
    x=t, y=x_true[:, 0],
    mode='lines',
    name='θ Real',
    line=dict(color='#1f77b4', width=2)
))
fig_theta.add_trace(go.Scatter(
    x=t, y=x_est_pf[:, 0],
    mode='lines',
    name='PF-RHONN θ',
    line=dict(color='#d62728', width=2, dash='dash')
))
fig_theta.update_layout(
    title=dict(
        text='Ángulo',
        x=0.5,
        xanchor='center',
        font=dict(size=thesis_config['title_size'], family=thesis_config['font_family'])
    ),
    xaxis=dict(title='Tiempo (s)', gridcolor=thesis_config['gridcolor'], showgrid=True),
    yaxis=dict(title='θ (rad)', gridcolor=thesis_config['gridcolor'], showgrid=True),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor=thesis_config['plot_bgcolor'],
    paper_bgcolor=thesis_config['paper_bgcolor'],
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5, font=dict(size=12))
)
fig_theta.show()

# ω trajectory figure
fig_omega = go.Figure()
fig_omega.add_trace(go.Scatter(
    x=t, y=x_true[:, 1],
    mode='lines',
    name='ω Real',
    line=dict(color='#1f77b4', width=2)
))
fig_omega.add_trace(go.Scatter(
    x=t, y=x_est_pf[:, 1],
    mode='lines',
    name='PF-RHONN ω',
    line=dict(color='#d62728', width=2, dash='dash')
))
fig_omega.update_layout(
    title=dict(
        text='Velocidad Angular',
        x=0.5,
        xanchor='center',
        font=dict(size=thesis_config['title_size'], family=thesis_config['font_family'])
    ),
    xaxis=dict(title='Tiempo (s)', gridcolor=thesis_config['gridcolor'], showgrid=True),
    yaxis=dict(title='ω (rad/s)', gridcolor=thesis_config['gridcolor'], showgrid=True),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor=thesis_config['plot_bgcolor'],
    paper_bgcolor=thesis_config['paper_bgcolor'],
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5, font=dict(size=12))
)
fig_omega.show()

print(f"\n✓ Separate state trajectory plots generated")



✓ Separate state trajectory plots generated


In [12]:
# ============================================================
# 8) IDENTIFICATION ERRORS
# ============================================================

errors_pf = x_true - x_est_pf

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Error de Identificación: θ', 'Error de Identificación: ω'),
    vertical_spacing=0.12
)

# θ error
fig.add_trace(go.Scatter(
    x=t, y=errors_pf[:, 0],
    mode='lines',
    name='Error θ',
    line=dict(color='#d62728', width=1.5),
    fill='tozeroy',
    fillcolor='rgba(214, 39, 40, 0.2)'
), row=1, col=1)

# ω error
fig.add_trace(go.Scatter(
    x=t, y=errors_pf[:, 1],
    mode='lines',
    name='Error ω',
    line=dict(color='#ff7f0e', width=1.5),
    fill='tozeroy',
    fillcolor='rgba(255, 127, 14, 0.2)',
    showlegend=False
), row=2, col=1)

# Zero reference lines
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=2, col=1)

fig.update_xaxes(title_text="Tiempo (s)", row=2, col=1,
                 gridcolor=thesis_config['gridcolor'], showgrid=True)
fig.update_xaxes(gridcolor=thesis_config['gridcolor'], showgrid=True, row=1, col=1)

fig.update_yaxes(title_text="Error (rad)", row=1, col=1,
                 gridcolor=thesis_config['gridcolor'], showgrid=True)
fig.update_yaxes(title_text="Error (rad/s)", row=2, col=1,
                 gridcolor=thesis_config['gridcolor'], showgrid=True)

fig.update_layout(
    height=600,
    title=dict(
        text='Errores de Identificación PF-RHONN',
        x=0.5,
        xanchor='center',
        font=dict(size=thesis_config['title_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor=thesis_config['plot_bgcolor'],
    paper_bgcolor=thesis_config['paper_bgcolor'],
    showlegend=False,
    hovermode='x unified'
)

fig.show()

print(f"\n✓ Identification error plots generated")


✓ Identification error plots generated


In [13]:
# ============================================================
# 9) ERROR METRICS COMPARISON
# ============================================================

# Prepare data for bar chart
metrics_names = ['RMSE', 'MAE', 'NRMSE']
states = ['θ', 'ω']

fig = go.Figure()

# RMSE
fig.add_trace(go.Bar(
    name='RMSE',
    x=states,
    y=[metrics_pf['RMSE'][0], metrics_pf['RMSE'][1]],
    marker_color='#1f77b4',
    text=[f"{metrics_pf['RMSE'][0]:.6f}", f"{metrics_pf['RMSE'][1]:.6f}"],
    textposition='outside',
    textfont=dict(size=11)
))

# MAE
fig.add_trace(go.Bar(
    name='MAE',
    x=states,
    y=[metrics_pf['MAE'][0], metrics_pf['MAE'][1]],
    marker_color='#ff7f0e',
    text=[f"{metrics_pf['MAE'][0]:.6f}", f"{metrics_pf['MAE'][1]:.6f}"],
    textposition='outside',
    textfont=dict(size=11)
))

# NRMSE
fig.add_trace(go.Bar(
    name='NRMSE',
    x=states,
    y=[metrics_pf['NRMSE'][0], metrics_pf['NRMSE'][1]],
    marker_color='#2ca02c',
    text=[f"{metrics_pf['NRMSE'][0]:.4f}", f"{metrics_pf['NRMSE'][1]:.4f}"],
    textposition='outside',
    textfont=dict(size=11)
))

fig.update_layout(
    title=dict(
        text='Métricas de Error PF-RHONN por Estado',
        x=0.5,
        xanchor='center',
        font=dict(size=thesis_config['title_size'], family=thesis_config['font_family'])
    ),
    xaxis_title='Estado',
    yaxis_title='Valor del Error',
    barmode='group',
    height=500,
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor=thesis_config['plot_bgcolor'],
    paper_bgcolor=thesis_config['paper_bgcolor'],
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.5
    ),
    yaxis=dict(gridcolor=thesis_config['gridcolor'], showgrid=True)
)

fig.show()

print(f"\n✓ Error metrics bar chart generated")
print(f"\nSummary:")
print(f"  RMSE Total: {metrics_pf['RMSE_total']:.6f}")
print(f"  MAE Total:  {metrics_pf['MAE_total']:.6f}")
print(f"  NRMSE Total: {metrics_pf['NRMSE_total']:.4f}")
print(f"  R² Total:   {metrics_pf['R2_total']:.4f}")


✓ Error metrics bar chart generated

Summary:
  RMSE Total: 0.002574
  MAE Total:  0.001852
  NRMSE Total: 0.0010
  R² Total:   1.0000


In [14]:
# ============================================================
# 10) PHASE PORTRAIT
# ============================================================

fig = go.Figure()

# True trajectory
fig.add_trace(go.Scatter(
    x=x_true[:, 0], 
    y=x_true[:, 1],
    mode='lines',
    name='Trayectoria Real',
    line=dict(color='#1f77b4', width=2),
    hovertemplate='θ: %{x:.3f} rad<br>ω: %{y:.3f} rad/s<extra></extra>'
))

# PF-RHONN trajectory
fig.add_trace(go.Scatter(
    x=x_est_pf[:, 0], 
    y=x_est_pf[:, 1],
    mode='lines',
    name='PF-RHONN',
    line=dict(color='#d62728', width=2, dash='dash'),
    hovertemplate='θ: %{x:.3f} rad<br>ω: %{y:.3f} rad/s<extra></extra>'
))

# Starting point
fig.add_trace(go.Scatter(
    x=[x_true[0, 0]], 
    y=[x_true[0, 1]],
    mode='markers',
    name='Inicio',
    marker=dict(color='green', size=12, symbol='circle'),
    showlegend=True
))

# Ending point
fig.add_trace(go.Scatter(
    x=[x_true[-1, 0]], 
    y=[x_true[-1, 1]],
    mode='markers',
    name='Fin',
    marker=dict(color='red', size=12, symbol='square'),
    showlegend=True
))

fig.update_layout(
    title=dict(
        text='Retrato de Fase: Péndulo Simple (θ vs ω)',
        x=0.5,
        xanchor='center',
        font=dict(size=thesis_config['title_size'], family=thesis_config['font_family'])
    ),
    xaxis=dict(
        title='Ángulo θ (rad)',
        gridcolor=thesis_config['gridcolor'],
        showgrid=True,
        zeroline=True,
        zerolinecolor='gray',
        zerolinewidth=1
    ),
    yaxis=dict(
        title='Velocidad Angular ω (rad/s)',
        gridcolor=thesis_config['gridcolor'],
        showgrid=True,
        zeroline=True,
        zerolinecolor='gray',
        zerolinewidth=1
    ),
    height=600,
    width=700,
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor=thesis_config['plot_bgcolor'],
    paper_bgcolor=thesis_config['paper_bgcolor'],
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.5
    ),
    hovermode='closest'
)

fig.show()

print(f"\n✓ Phase portrait generated")


✓ Phase portrait generated


In [15]:
# ============================================================
# 11) CONTROL INPUT AND DETAILED VIEW
# ============================================================

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Entrada de Control τ(t)', 'Ángulo θ(t) - Vista Detallada', 'Velocidad Angular ω(t) - Vista Detallada'),
    vertical_spacing=0.10
)

# Control input
fig.add_trace(go.Scatter(
    x=t, y=u_hist[:, 0],
    mode='lines',
    name='Torque τ',
    line=dict(color='#9467bd', width=2),
    fill='tozeroy',
    fillcolor='rgba(148, 103, 189, 0.2)'
), row=1, col=1)

# Detailed view: Time window [5s, 8s]
time_start, time_end = 5.0, 8.0
idx_start = int(time_start / dt)
idx_end = int(time_end / dt)

t_zoom = t[idx_start:idx_end]

# θ detailed
fig.add_trace(go.Scatter(
    x=t_zoom, y=x_true[idx_start:idx_end, 0],
    mode='lines',
    name='θ Real',
    line=dict(color='#1f77b4', width=2),
    showlegend=False
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=t_zoom, y=x_est_pf[idx_start:idx_end, 0],
    mode='lines',
    name='PF-RHONN θ',
    line=dict(color='#d62728', width=2, dash='dash'),
    showlegend=False
), row=2, col=1)

# ω detailed
fig.add_trace(go.Scatter(
    x=t_zoom, y=x_true[idx_start:idx_end, 1],
    mode='lines',
    name='ω Real',
    line=dict(color='#1f77b4', width=2),
    showlegend=False
), row=3, col=1)

fig.add_trace(go.Scatter(
    x=t_zoom, y=x_est_pf[idx_start:idx_end, 1],
    mode='lines',
    name='PF-RHONN ω',
    line=dict(color='#d62728', width=2, dash='dash'),
    showlegend=False
), row=3, col=1)

fig.update_xaxes(title_text="Tiempo (s)", gridcolor=thesis_config['gridcolor'], showgrid=True)
fig.update_yaxes(title_text="τ (N·m)", row=1, col=1, gridcolor=thesis_config['gridcolor'], showgrid=True)
fig.update_yaxes(title_text="θ (rad)", row=2, col=1, gridcolor=thesis_config['gridcolor'], showgrid=True)
fig.update_yaxes(title_text="ω (rad/s)", row=3, col=1, gridcolor=thesis_config['gridcolor'], showgrid=True)

fig.update_layout(
    height=800,
    title=dict(
        text='Entrada de Control y Vista Detallada de Trayectoria',
        x=0.5,
        xanchor='center',
        font=dict(size=thesis_config['title_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor=thesis_config['plot_bgcolor'],
    paper_bgcolor=thesis_config['paper_bgcolor'],
    showlegend=False,
    hovermode='x unified'
)

fig.show()

print(f"\n✓ Control input and detailed view generated")
print(f"\nDetailed view window: [{time_start}s, {time_end}s]")


✓ Control input and detailed view generated

Detailed view window: [5.0s, 8.0s]


## 📋 Notebook Summary

### System Configuration
- **System**: Simple Pendulum with damping and external torque
- **States**: 2 states [θ (angle), ω (angular velocity)]
- **Control**: Single torque input τ
- **Discretization**: RK4 (4th order Runge-Kutta)

### Identification Method
- **Algorithm**: PF-RHONN (Particle Filter + Recurrent High Order Neural Network)
- **Particles**: 1000 particles per neuron
- **Features**: Neuron-specific (5 for θ, 6 for ω)
- **Configuration**: Series (PF uses measurements as input)

### RHONN Architecture
**Neuron 0 (θ)**: 5 features
- Sigmoid of ω, ω², θ, θ·ω, bias

**Neuron 1 (ω)**: 6 features
- Sigmoid of θ, ω, ω², τ, θ·ω, bias

### Visualizations Generated
1. **State Trajectories**: Comparison of true vs identified states
2. **Identification Errors**: Time-series error analysis
3. **Error Metrics**: RMSE, MAE, NRMSE comparison by state
4. **Phase Portrait**: θ vs ω trajectory in phase space
5. **Control Input**: External torque and detailed trajectory views
6. **Comprehensive Summary**: 4-panel overview of all key results

### Key Features
- Non-Gaussian noise modeling (Student's t-distribution)
- NaN prevention strategies
- Systematic resampling with regularization
- Log-likelihood computation for numerical stability
- Comprehensive error metrics (RMSE, MAE, NRMSE, R²)

---

**Ready for execution!** Run cells in order from top to bottom.

In [16]:
# ============================================================
# FINAL COMPREHENSIVE SUMMARY VISUALIZATION
# ============================================================

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Comparación de Trayectorias de Estado',
        'Errores de Identificación', 
        'Métricas de Error (RMSE, MAE)',
        'Retrato de Fase (θ vs ω)'
    ),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"type": "bar"}, {"secondary_y": False}]],
    vertical_spacing=0.15,
    horizontal_spacing=0.12
)

# 1. State trajectories (both states on same plot)
fig.add_trace(go.Scatter(
    x=t, y=x_true[:, 0], mode='lines', name='θ Real',
    line=dict(color='#1f77b4', width=1.5)
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=t, y=x_est_pf[:, 0], mode='lines', name='PF θ',
    line=dict(color='#d62728', width=1.5, dash='dash')
), row=1, col=1)

# 2. Identification errors
errors_pf = x_true - x_est_pf
fig.add_trace(go.Scatter(
    x=t, y=errors_pf[:, 0], mode='lines', name='Error θ',
    line=dict(color='#d62728', width=1),
    fill='tozeroy', fillcolor='rgba(214, 39, 40, 0.2)',
    showlegend=False
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=t, y=errors_pf[:, 1], mode='lines', name='Error ω',
    line=dict(color='#ff7f0e', width=1),
    fill='tozeroy', fillcolor='rgba(255, 127, 14, 0.2)',
    showlegend=False
), row=1, col=2)

# 3. Error metrics bar chart
states_short = ['θ', 'ω']
fig.add_trace(go.Bar(
    name='RMSE', x=states_short, 
    y=[metrics_pf['RMSE'][0], metrics_pf['RMSE'][1]],
    marker_color='#1f77b4',
    showlegend=True
), row=2, col=1)
fig.add_trace(go.Bar(
    name='MAE', x=states_short,
    y=[metrics_pf['MAE'][0], metrics_pf['MAE'][1]],
    marker_color='#ff7f0e',
    showlegend=True
), row=2, col=1)

# 4. Phase portrait
fig.add_trace(go.Scatter(
    x=x_true[:, 0], y=x_true[:, 1], mode='lines',
    name='Real', line=dict(color='#1f77b4', width=1.5),
    showlegend=False
), row=2, col=2)
fig.add_trace(go.Scatter(
    x=x_est_pf[:, 0], y=x_est_pf[:, 1], mode='lines',
    name='PF', line=dict(color='#d62728', width=1.5, dash='dash'),
    showlegend=False
), row=2, col=2)

# Start and end markers
fig.add_trace(go.Scatter(
    x=[x_true[0, 0]], y=[x_true[0, 1]], mode='markers',
    marker=dict(color='green', size=10, symbol='circle'),
    name='Inicio', showlegend=False
), row=2, col=2)

# Update axes
fig.update_xaxes(title_text="Tiempo (s)", row=1, col=1, gridcolor='rgba(128,128,128,0.2)', showgrid=True)
fig.update_xaxes(title_text="Tiempo (s)", row=1, col=2, gridcolor='rgba(128,128,128,0.2)', showgrid=True)
fig.update_xaxes(title_text="Estado", row=2, col=1)
fig.update_xaxes(title_text="θ (rad)", row=2, col=2, gridcolor='rgba(128,128,128,0.2)', showgrid=True)

fig.update_yaxes(title_text="Estados", row=1, col=1, gridcolor='rgba(128,128,128,0.2)', showgrid=True)
fig.update_yaxes(title_text="Error", row=1, col=2, gridcolor='rgba(128,128,128,0.2)', showgrid=True)
fig.update_yaxes(title_text="Valor del Error", row=2, col=1, gridcolor='rgba(128,128,128,0.2)', showgrid=True)
fig.update_yaxes(title_text="ω (rad/s)", row=2, col=2, gridcolor='rgba(128,128,128,0.2)', showgrid=True)

# Add zero lines to error plot
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=2)

fig.update_layout(
    height=800,
    title=dict(
        text='Identificación del Péndulo Simple - Resumen Comprehensivo (PF-RHONN)',
        x=0.5,
        xanchor='center',
        font=dict(size=16, family='Computer Modern')
    ),
    font=dict(size=12, family='Computer Modern'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.25,
        font=dict(size=11)
    ),
    barmode='group',
    showlegend=True
)

fig.show()

print("\n" + "="*70)
print("📊 COMPREHENSIVE SUMMARY COMPLETE")
print("="*70)
print(f"\n✅ All visualizations generated successfully!")
print(f"\n🎯 Final Results:")
print(f"  • RMSE Total: {metrics_pf['RMSE_total']:.6f}")
print(f"  • MAE Total:  {metrics_pf['MAE_total']:.6f}")
print(f"  • NRMSE Total: {metrics_pf['NRMSE_total']:.4f}")
print(f"  • R² Score: {metrics_pf['R2_total']:.4f} ({metrics_pf['R2_total']*100:.2f}%)")
print(f"  • Mean training time: {pf_mean_time*1000:.2f} ms per iteration")
print(f"\n💡 Identification Quality: {'🌟 Excellent' if metrics_pf['R2_total'] > 0.95 else '✓ Good' if metrics_pf['R2_total'] > 0.85 else 'Acceptable'}")
print("\n" + "="*70)


📊 COMPREHENSIVE SUMMARY COMPLETE

✅ All visualizations generated successfully!

🎯 Final Results:
  • RMSE Total: 0.002574
  • MAE Total:  0.001852
  • NRMSE Total: 0.0010
  • R² Score: 1.0000 (100.00%)
  • Mean training time: 0.15 ms per iteration

💡 Identification Quality: 🌟 Excellent

